> **Superseded, and misleading if read on its own — kept for the record.**
>
> **The oracle in this notebook is handed the answer.** `match_positions` is
> computed classically, in Python, before any circuit is built, and the oracle
> is then constructed to flip the phase of positions that are already known. It
> runs, and the histogram peaks in the right place, but what it demonstrates is
> Grover *amplification* — not string matching. It cannot support any claim
> about search complexity, because the O(N) classical scan has already happened
> by the time the first qubit is allocated.
>
> The replacement is
> [`notebooks/03_grover_comparator.ipynb`](../../notebooks/03_grover_comparator.ipynb),
> whose oracle is given only the text and the pattern and performs the
> comparison inside the circuit, in superposition over all candidate positions,
> uncomputing the window register back to |0⟩ afterwards so that the phase
> attaches to the position register alone (verified: leakage is exactly zero).
>
> Section 4.1 of `docs/final_report.md` records why this notebook was replaced.

---

# Grover-based String Matching — Starter Notebook

Goal: build an oracle that flips the phase of position `i` in a text if and only if
the pattern `P` matches the text starting at position `i`.

Start toy: 8-character text (3 position qubits), 2-character pattern.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram
import numpy as np

## 1. Toy setup

Text: 8 characters -> 3 position qubits (positions 0-7).
Pattern: 2 characters. For a first pass, pick a text/pattern where exactly ONE
position is a true match, to make verification simple.

In [ ]:
text = "ABABXYAB"   # 8 chars, positions 0-7
pattern = "XY"       # matches at position 4 only

match_positions = [i for i in range(len(text) - len(pattern) + 1)
                    if text[i:i+len(pattern)] == pattern]
print(f"Text: {text}")
print(f"Pattern: {pattern}")
print(f"True match position(s): {match_positions}")

## 2. Build the oracle

For this toy version, since we know the match position classically ahead of time,
start with a MARKED-STATE oracle (like standard Grover) to get the mechanics right —
flip the phase of the known match position(s).

Once this works end-to-end, replace this with a real comparator-based oracle that
checks character equality directly in-circuit (the actual project deliverable) —
that's the harder, more valuable version. Don't skip building this step-1 version first;
it isolates diffusion/iteration-count bugs from oracle-construction bugs.

In [ ]:
def build_marked_state_oracle(n_qubits, marked_positions):
    qc = QuantumCircuit(n_qubits)
    for pos in marked_positions:
        # flip phase of |pos> using X gates + multi-controlled Z + X gates
        bin_str = format(pos, f'0{n_qubits}b')
        for i, bit in enumerate(bin_str):
            if bit == '0':
                qc.x(i)
        qc.h(n_qubits - 1)
        qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
        qc.h(n_qubits - 1)
        for i, bit in enumerate(bin_str):
            if bit == '0':
                qc.x(i)
    return qc

n_pos_qubits = 3
oracle = build_marked_state_oracle(n_pos_qubits, match_positions)
oracle.draw('mpl')

## 3. Diffusion operator

In [ ]:
def build_diffusion(n_qubits):
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    qc.x(range(n_qubits))
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    qc.h(n_qubits - 1)
    qc.x(range(n_qubits))
    qc.h(range(n_qubits))
    return qc

diffusion = build_diffusion(n_pos_qubits)
diffusion.draw('mpl')

## 4. Full circuit — superposition, iterate oracle+diffusion, measure

In [ ]:
n_iterations = 2  # optimal ~ (pi/4)*sqrt(N/M), tune based on N=8, M=len(match_positions)

qc = QuantumCircuit(n_pos_qubits, n_pos_qubits)
qc.h(range(n_pos_qubits))

for _ in range(n_iterations):
    qc.compose(oracle, inplace=True)
    qc.compose(diffusion, inplace=True)

qc.measure(range(n_pos_qubits), range(n_pos_qubits))

sim = AerSimulator()
compiled = transpile(qc, sim)
result = sim.run(compiled, shots=1024).result()
counts = result.get_counts()
plot_histogram(counts)

## 5. Verify

Check the highest-probability outcome corresponds to the true match position
(remember Qiskit's bit ordering — verify against `match_positions` carefully).

In [ ]:
top_result = max(counts, key=counts.get)
print(f"Most measured bitstring: {top_result} -> position {int(top_result, 2)}")
print(f"Expected match position(s): {match_positions}")

## Next steps

1. Confirm this marked-state version works cleanly first.
2. Build the real comparator-based oracle: encode text characters into qubits,
   check character-by-character equality in-circuit, combine into one phase flip
   (this replaces `build_marked_state_oracle` with the actual project deliverable).
3. Scale pattern/text length gradually, log where gate count/depth becomes the bottleneck.